<a href="https://colab.research.google.com/github/julmiha25-sys/Python/blob/main/%D0%9A%D0%B5%D0%B9%D1%811.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import os, re, glob, sys, csv # Работа с ОС, регул. выраж., поиск файлов, сист. пар-ми, csv-файлами

def process_files(input_path, output_path):
    sys.path.append(os.path.join(os.getcwd(), '..', input_path))
    if not os.path.exists(input_path) or not os.path.exists(output_path):
        print(f"❌ Папки '{input_path}' или '{output_path}' не существует")
        return None

    filter_files = r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d+\.csv' # шаблон регулярных выражений
    files_cvs = glob.glob(os.path.join(input_path, '*.csv')) # поиск файлов по шаблону
    files = [f for f in files_cvs if re.match(filter_files, os.path.basename(f))] # фильтрация файлов csv

    if not files_cvs:
        print(f"❌ В папке '{input_path}' не найдено CSV файлов")
        return None

    if len(files_cvs) > len(files):
        print(f"❌ В папке '{input_path}' присутствуют CSV файлы, но не в нужном формате.")
        print(f"    Ожидаемый формат: ГГГГ-ММ-ДД-ЧЧ-ММ-idмагазина.csv \n")
        if not files:
            return None

    print(f"✅ Загруженные файлы в кол-ве {len(files)}:")
    for f in files:
        print(f" - {os.path.basename(f)}")

    output_file = os.path.join(output_path, 'combined_data.csv')
    all_rows = [] # Список всех строк csv-файлов
    headers = None # Заголовок
    total_rows = 0  # Счетчик кол-ва строк из успешно обработанных файлов
    processed_files = 0  # Счетчик кол-ва успешно обработанных файлов
    skipped_files = []  # Список с кортежами с инф-ей о необработанных файлах

    print(f"\n🔄 Файлы для обработки:")
    first_valid_index = -1  # Индекс первого валидного файла (-1 - еще не найден)

    for idx, file_path in enumerate(files): # Поиск первого валидного файла для взятия заголовка
        file_name = os.path.basename(file_path)

        for encoding in ['utf-8', 'cp1251']: # По умолчанию работаем с 2-я кодировками
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    reader = csv.reader(f, quotechar='"') # Создание объекта для построчного чтения csv-файла
                    current_headers = next(reader) # Строка заголовка

                    file_rows = []  # Список строк всего файла - для проверки отсут-я строк после заголовка
                    for row in reader: # Чтение со 2-й строки
                        file_rows.append(row)

                    if len(file_rows) == 0:
                        skipped_files.append((idx + 1, file_name, "Файл содержит только заголовки, пропускаем"))
                        break

                    headers = current_headers  # Установка заголовка
                    first_valid_index = idx # Выбираем индекс валидного файла
                    print(f"    Колонки заголовка: {', '.join(current_headers)}")

                    all_rows.extend(file_rows) # Добавление строк всего файла в общий список
                    total_rows += len(file_rows) # Наращивание общего кол-ва строк
                    processed_files += 1 # Наращивание кол-ва обраб. файлов
                    enc_suffix = f" ({encoding})" if encoding != 'utf-8' else "" # utf-8 по умолч., если cp1251 - то коммент.
                    print(f"  {idx + 1}. ☑️ {file_name} - {len(file_rows):,} строк{enc_suffix}")
                    break

            except StopIteration: # Если файл пустой
                skipped_files.append((idx + 1, file_name, "Файл пустой, пропускаем"))
                break
            except UnicodeDecodeError: # Если кодировка не utf-8 и не cp1251
                continue
            except Exception as e:  # Прочие исключения
                skipped_files.append((idx + 1, file_name, f"Ошибка обработки файла: {str(e)[:50]}"))
                break

        if first_valid_index != -1: # Если найден первый валидный файл - прерываем
            break

    if first_valid_index == -1: # Нет валидных файлов
        for num, fname, reason in skipped_files: # Распаковка кортежа
            print(f"  {num}. ⭕ {fname} - {reason}")
        print("\n❌ Не удалось найти ни одного файла с данными")
        return None

    for idx in range(first_valid_index + 1, len(files)): # Обработка файлов после первого валидного
        file_path = files[idx]
        file_name = os.path.basename(file_path)
        file_processed = False # Флаг - файл еще не обработан

        for encoding in ['utf-8', 'cp1251']: # Обработка ост. файлов после первого валидного
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    reader = csv.reader(f, quotechar='"')
                    current_headers = next(reader)

                    file_rows = []
                    for row in reader: # Читаем все строки из файла
                        file_rows.append(row)

                    if len(file_rows) == 0:
                        print(f"  {idx + 1}. ⭕ {file_name} - Файл содержит только заголовки")
                        file_processed = True
                        break

                    if current_headers != headers: # Колонки не соотв. заголовку первого валид. файла
                        print(f"  {idx + 1}. ⭕ {file_name} - Файл содержит несовместимые колонки")
                        file_processed = True
                        break

                    all_rows.extend(file_rows) # Добавляем все строки в общий список
                    total_rows += len(file_rows) # Наращиваем кол-во обработ. строк
                    processed_files += 1 # Наращиваем кол-во обраб. файлов
                    enc_suffix = f" ({encoding})" if encoding != 'utf-8' else ""
                    print(f"  {idx + 1}. ☑️ {file_name} - {len(file_rows):,} строк{enc_suffix}")
                    file_processed = True # Флаг - обр-ка текущ. файла завершена
                    break

            except UnicodeDecodeError: # Переходим к следующей кодировке
                continue
            except StopIteration: # Файл пустой
                print(f"  {idx + 1}. ⭕ {file_name} - Файл пустой")
                file_processed = True
                break
            except Exception as e: # Прочие исключения (права доступа, повреж., ошибка формата, памяти)
                print(f"  {idx + 1}. ⭕ {file_name} - Ошибка обработки файла: {str(e)[:50]}")
                file_processed = True
                break

        if not file_processed: # Другая кодировка (не utf-8 и не cp1251)
            print(f"  {idx + 1}. ⭕ {file_name} - Не удалось прочитать файл")

    for num, fname, reason in skipped_files: # Распаковка кортежа пропущ. файлов
        if num <= first_valid_index + 1:
            print(f"  {num}. ⭕ {fname} - {reason}")

    if not all_rows: # Ни один файл не удалось прочитать
        print("\n❌ Не удалось прочитать данные")
        return None

    try:
        with open(output_file, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)  # Создание объекта для записи вых. файла
            writer.writerow(headers) # Создание строки заголовка
            writer.writerows(all_rows) # Запись всех строк
        print(f"\n✅ Файл создан: {output_file}")
        print(f" - Всего строк: {total_rows}")
        print(f" - Файлов обработано: {processed_files} из {len(files)}")
    except Exception as e:
        print(f"\n❌ Ошибка при сохранении файла: {e}")
        return None


process_files('/content/Input', '/content/Output')

✅ Загруженные файлы в кол-ве 3:
 - 2025-01-01-12-50-1.csv
 - 2025-01-05-13-50-9.csv
 - 2025-05-05-13-50-1.csv

🔄 Файлы для обработки:
    Колонки заголовка: expr, Therapy
  1. ☑️ 2025-01-01-12-50-1.csv - 60 строк
  2. ☑️ 2025-01-05-13-50-9.csv - 60 строк
  3. ⭕ 2025-05-05-13-50-1.csv - Файл содержит несовместимые колонки

✅ Файл создан: /content/Output/combined_data.csv
 - Всего строк: 120
 - Файлов обработано: 2 из 3


# Новый раздел